* `TOIs_GaiaDR3_xmatch_new.csv` uses ID-based matching which is superior than `TOIs_GaiaDR3_xmatch.csv` which relies on position-based matching
* In `TOIs_GaiaDR3_xmatch_new.csv`, not all TIC ID has an associated Gaia ID.
* There is also significant difference in Teff from TIC and Gaia GSPPhot.

In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

# TOI catalog

In [ ]:
from wakai.catalog import get_tois

tois = get_tois(clobber=False)
# tois.columns.tolist()
tois.shape

In [ ]:
tois.tail()

In [ ]:
tois[tois.Comments.apply(lambda x: 'young' in str(x))][['TOI','Comments']].values

# get TIC catalog data containing Gaia ID for exominer TCEs

# cross-match TOI TICID with TIC catalog then Gaia catalog using `astroquery.mast`

In [ ]:
from astroquery.mast import Catalogs
import numpy as np

# Query the TIC catalog for these IDs
# We specifically want the 'GAIA' column which contains the Gaia DR2/DR3 source_id
tic_data = Catalogs.query_criteria(catalog="TIC", ID=tois['TIC ID'].unique())

In [ ]:
tic_data.columns #why no RV?

In [ ]:
tic_df = tic_data.to_pandas()[['ID', 'ra', 'dec', 'plx', 'pmRA', 'pmDEC', 'Teff']].astype(float)
# Gaia DR3 source_id is a ~19-digit integer beyond float64's 2**53 exact-integer
# range. `Table.to_pandas()` silently loses precision for *masked* int64 columns
# (it round-trips through float64 internally), so build this column directly
# from the raw astropy column instead of trusting the pandas conversion.
tic_df['GAIA'] = pd.array(
    [int(gid) if gid is not np.ma.masked else pd.NA for gid in tic_data['GAIA']],
    dtype='Int64',
)
tic_df = tic_df[['ID', 'GAIA', 'ra', 'dec', 'plx', 'pmRA', 'pmDEC', 'Teff']]
tic_df.shape

https://gaia.aip.de/metadata/gaiadr3/gaia_source/

In [ ]:
from astroquery.gaia import Gaia

# Extract unique Gaia IDs and remove masked/null values
gaia_ids = [int(gid) for gid in tic_data['GAIA'] if gid is not np.ma.masked]

# Format the IDs for the SQL 'IN' clause
ids_str = ",".join(map(str, gaia_ids))

# ADQL query to get source_id and radial_velocity
query = f"""
SELECT source_id, radial_velocity, radial_velocity_error, teff_gspphot
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_str})
"""

# Launch the job
job = Gaia.launch_job_async(query)
results = job.get_results()

In [ ]:
import pandas as pd

rv_df = results.to_pandas().rename({
    'SOURCE_ID': 'GAIA',
    }, axis=1
                                  )
rv_df['GAIA'] = rv_df['GAIA'].astype('Int64')
for col in ['radial_velocity', 'radial_velocity_error', 'teff_gspphot']:
    rv_df[col] = rv_df[col].astype(float)
tic_df = pd.merge(tic_df, rv_df, on='GAIA')
tic_df['TIC ID'] = tic_df['ID'].astype(int).copy()
tic_df.head()

In [ ]:
matches = pd.merge(tois, tic_df, on='TIC ID')

In [ ]:
matches = matches.rename({
    'GAIA': 'GaiaDR3',
    'plx': 'parallax',
    'pmRA': 'pmra',
    'pmDEC': 'pmdec',
    'Teff': 'Teff_TIC',
    'teff_gspphot': 'Teff_gaia'
}, axis=1)

In [ ]:
diff = matches['Teff_TIC']-matches['Teff_gaia']
ax = diff.hist(bins=100)
ax.set_xlim(-1000,1000)

In [ ]:
diff.std()

In [ ]:
matches.to_csv('../data/TOIs_GaiaDR3_xmatch_id.csv', index=False)

In [ ]:
matches.GaiaDR3.unique().shape, tois.shape

# cross-match TOI position with Gaia DR3 using `astroquery.xmatch`

In [ ]:
import astropy.units as u
from astroquery.xmatch import XMatch
import pandas as pd

gaia_table = XMatch.query(
    cat1=open('../data/TOIs.csv'),
    cat2="vizier:I/355/gaiadr3",  # The Vizier identifier for Gaia DR3
    max_distance=5 * u.arcsec,    # Set the maximum matching radius
    colRA1="ra_deg",  # Column name for Right Ascension in TIC results
    colDec1="dec_deg" # Column name for Declination in TIC results
)
gaia_df = gaia_table.to_pandas()

# results to 6237 matches
matches = gaia_df.copy()

In [ ]:
matches.TOI.duplicated().sum()

In [ ]:
# Apply magnitude filtering criteria
# For main-sequence stars, the difference between TESS T and Gaia G is around 0–0.5 magnitudes
# Redder stars might have T - G values closer to 1–2 magnitudes.
# For giants or highly reddened stars, the difference could be larger.
if False:
    # results to 5971 matches
    matches = []
    for i, row in gaia_df.iterrows():
        teff=row['Stellar Eff Temp (K)']
        if teff<3500:
            magnitude_threshold = 2
        elif (teff>3500) and (teff<6000):
            magnitude_threshold = 1
        else:
            magnitude_threshold = 1.5
        if abs(row['Gmag'] - row['TESS Mag']) < magnitude_threshold:
            matches.append(row)
    matches = pd.concat(matches, axis=1).T
if False:
    # results to 5700 matches
    magnitude_threshold = 2
    matches = gaia_df[abs(gaia_df['Gmag'] - gaia_df['TESS Mag']) < magnitude_threshold]
if False:
    # results to 4878 matches
    teff_threshold = 1000
    matches = gaia_df[abs(gaia_df['Teff'] - gaia_df['Stellar Eff Temp (K)']) < teff_threshold]

# Keep the nearest Gaia DR3 candidate (smallest angDist) per TOI instead of
# saving every candidate within the 5" search radius; the magnitude/Teff
# heuristics above were exploratory and were never adopted.
matches = matches.sort_values('angDist').drop_duplicates(subset='TOI', keep='first').sort_index()
matches.tail()

In [ ]:
matches.shape

In [ ]:
matches['GaiaDR3'] = matches['DR3Name'].apply(lambda x: int(x.split()[-1]))
matches.to_csv('../data/TOIs_GaiaDR3_xmatch_position.csv', index=False)

In [ ]:
idx = tois.TOI.isin(matches.TOI.unique())
tois[~idx].shape

In [ ]:
tois[~idx].to_csv('../data/TOIs_GaiaDR3_xmatch_missing.csv', index=False)